## Упражнение 02. Соединение (JOIN)

In [3]:
import pandas as pd
import sqlite3

### 1. Создаем соединение с базой данных с помощью библиотеки sqlite3

In [5]:
conn=sqlite3.connect('../data/checking-logs.sqlite')

### 2. Одним запросом создаем в базе новую таблицу datamart, объединив таблицы pageviews и checker.
#### - В таблице должны быть колонки: "uid", "labname", "first_commit_ts", "first_view_ts";
#### - "first_commit_ts" - новое имя колонки "timestamp" из таблицы checker; это первая отправка для конкретной лабораторной и пользователя;
#### - "first_view_ts" - первая временная метка посещения пользователем ленты (newsfeed) из pageviews;
#### - Фильтр status = 'ready' должен сохраняться;
#### - Фильтр numTrials = 1 должен сохраняться;
#### - "labnames" должен входить в список: laba04, laba04s, laba05, laba06, laba06s, и project1;
#### - В таблицу должны попадать только пользователи (uid вида user_*), администраторы - исключаются;
#### - Колонки "first_commit_ts" и "first_view_ts" должны иметь тип datetime64[ns].

In [6]:
query = """
        create table if not exists datamart as
        select checker.uid, checker.labname, checker.timestamp as first_commit_ts,
        pageviews.datetime as first_view_ts from checker
        left join pageviews on pageviews.uid = checker.uid
        where checker.uid like 'user_%' and
        status = 'ready' and numTrials = 1 and
        labname in ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
        and (pageviews.datetime = (select min(datetime) from pageviews where checker.uid = pageviews.uid) or pageviews.datetime is null)
        """

conn.execute(query)

df = pd.io.sql.read_sql('select * from datamart', conn, parse_dates=['first_commit_ts', 'first_view_ts'])
df

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,NaT
1,user_4,laba04,2020-04-17 11:33:17.366400,NaT
2,user_4,laba04s,2020-04-17 11:48:41.992466,NaT
3,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
4,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
...,...,...,...,...
135,user_23,laba06,2020-05-21 08:34:10.517205,NaT
136,user_19,laba06s,2020-05-21 13:27:06.705881,2020-04-21 20:30:38.034966
137,user_23,laba06s,2020-05-21 14:29:15.709568,NaT
138,user_17,laba06,2020-05-21 15:21:31.567615,2020-04-18 10:56:55.833899


### 3. С помощью методов Pandas создаем два DataFrame: test и control.
#### - test содержит пользователей с заполненными значениями в "first_view_ts";
#### - control содержит пользователей с пропущенными значениями в "first_view_ts";
#### - Заменил пропуски в control средним значением "first_view_ts" среди пользователей из test.
### Сохранил обе таблицы в базе.

In [7]:
test=df[df['first_view_ts'].notnull()]
control=df[df['first_view_ts'].isnull()]
control.loc[:, 'first_view_ts'] = test['first_view_ts'].mean()
test.to_sql("test", conn, index=False)
control.to_sql("control", conn, index=False)

ValueError: Table 'test' already exists.

In [8]:
pd.io.sql.read_sql("pragma table_info(test)",conn)

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


In [9]:
pd.io.sql.read_sql("pragma table_info(control)",conn)

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


### 4. Закрываем соединение

In [10]:
conn.close()